** just generate 5 examples for train and test

# Cell 1: Imports and Configurations

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()
import torch
import torch.nn as nn
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

# --- SPEED OPTIMIZATIONS ---
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# --- MODEL PATH ---
BEST_MODEL = "best_multilayer_injector.pt"

# --- LOAD LLAMA (FROZEN) ---
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
HF_TOKEN = os.getenv("HF_TOKEN")

tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id, device_map="auto", torch_dtype=torch.bfloat16, attn_implementation="sdpa", token=HF_TOKEN
)
model.eval()
model.requires_grad_(False)
device = model.device

Loading weights: 100%|██████████| 291/291 [00:02<00:00, 142.20it/s]


# Cell 2: Injector and Data Load (choosing 5 examples)

In [4]:
# --- MULTI-LAYER ARCHITECTURE & SAFE LOAD ---
class SingleLayerInjector(nn.Module):
    def __init__(self, dim=4096, bottleneck_dim=256, dropout_p=0.05):
        super().__init__()
        self.norm_q = nn.LayerNorm(dim)
        self.norm_k = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=8, batch_first=True, dropout=dropout_p)
        self.proj = nn.Sequential(
            nn.Linear(dim, bottleneck_dim),
            nn.GELU(),
            nn.Dropout(dropout_p),
            nn.Linear(bottleneck_dim, dim)
        )
        self.final_dropout = nn.Dropout(dropout_p)
        self.alpha_gate = nn.Sequential(
            nn.Linear(dim, 128),
            nn.GELU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
        self.max_alpha = 5.0

    def forward(self, query, keys, padding_mask):
        q_norm = self.norm_q(query)
        k_norm = self.norm_k(keys)
        attn_out, _ = self.attn(q_norm, k_norm, k_norm, key_padding_mask=padding_mask, need_weights=False)
        bottleneck_out = self.final_dropout(self.proj(attn_out))
        dynamic_alpha = self.alpha_gate(query) * self.max_alpha
        return dynamic_alpha * bottleneck_out

class MultiLayerInjector(nn.Module):
    def __init__(self, dim=4096, bottleneck_dim=256, dropout_p=0.05):
        super().__init__()
        self.layer_8 = SingleLayerInjector(dim, bottleneck_dim, dropout_p)
        self.layer_16 = SingleLayerInjector(dim, bottleneck_dim, dropout_p)
        self.layer_24 = SingleLayerInjector(dim, bottleneck_dim, dropout_p)

injector = MultiLayerInjector(dim=4096, bottleneck_dim=256, dropout_p=0.0).to(device, dtype=torch.bfloat16)

print(f"Loading best weights from {BEST_MODEL}...")

# Scrub the "_orig_mod." prefix left by torch.compile
raw_state_dict = torch.load(BEST_MODEL, map_location=device, weights_only=True)
clean_state_dict = {}
for key, value in raw_state_dict.items():
    clean_key = key.replace('_orig_mod.', '') if key.startswith('_orig_mod.') else key
    clean_state_dict[clean_key] = value

injector.load_state_dict(clean_state_dict)
injector.eval()

# --- LOAD DATASETS & VECTORS ---
print("Loading Datasets & Vectors...")
train_df = pd.read_csv("train_tasks_expanded.csv")
test_df = pd.read_csv("test_tasks_expanded.csv")

train_df = train_df.dropna(subset=['target_response']).reset_index(drop=True)
test_df = test_df.dropna(subset=['target_response']).reset_index(drop=True)

train_vecs = torch.load("train_vectors_unified.pt", weights_only=False)
test_vecs = torch.load("test_vectors_unified.pt", weights_only=False)

train_sample = train_df.sample(n=10, random_state=42).reset_index(drop=True)
test_sample = test_df.sample(n=10, random_state=42).reset_index(drop=True)

# --- 6. GENERATION HOOKS (PREFILL ONLY) ---
ACTIVE_SPONGE_IDX = None
ACTIVE_MASK = None
ACTIVE_KV_8 = None
ACTIVE_KV_16 = None
ACTIVE_KV_24 = None

def create_generation_hook(layer_name):
    def hook(module, args, output):
        hs = output[0] if isinstance(output, tuple) else output
        seq_len = hs.shape[1]
        if seq_len > 1 and ACTIVE_SPONGE_IDX < seq_len:
            query = hs[:, ACTIVE_SPONGE_IDX, :].unsqueeze(1)

            if layer_name == "layer_8": injection = injector.layer_8(query, ACTIVE_KV_8, ACTIVE_MASK)
            elif layer_name == "layer_16": injection = injector.layer_16(query, ACTIVE_KV_16, ACTIVE_MASK)
            elif layer_name == "layer_24": injection = injector.layer_24(query, ACTIVE_KV_24, ACTIVE_MASK)

            hs_mod = hs.clone()
            hs_mod[:, ACTIVE_SPONGE_IDX, :] += injection.squeeze(1)
            return (hs_mod,) + output[1:] if isinstance(output, tuple) else hs_mod

        return output
    return hook

h8 = model.model.layers[8].register_forward_hook(create_generation_hook("layer_8"))
h16 = model.model.layers[16].register_forward_hook(create_generation_hook("layer_16"))
h24 = model.model.layers[24].register_forward_hook(create_generation_hook("layer_24"))

# --- 7. INFERENCE ENGINE ---
def generate_steered_response(row, vecs_dict):
    global ACTIVE_KV_8, ACTIVE_KV_16, ACTIVE_KV_24, ACTIVE_MASK, ACTIVE_SPONGE_IDX

    c_id = row['question_id']
    prompt = row['inj_prompt']

    data = vecs_dict[c_id]
    chunks = data['chunk_signals'].to(device)
    gl = data['global_last'].to(device)

    kv_8_list, kv_16_list, kv_24_list = [], [], []
    for l_idx, lst in zip([0, 1, 2], [kv_8_list, kv_16_list, kv_24_list]):
        mu = chunks[:, l_idx, 0, :]
        max_vec = chunks[:, l_idx, 2, :]
        layer_gl = gl[l_idx, :].unsqueeze(0)

        kv_seq = torch.cat([mu, max_vec, layer_gl], dim=0)
        lst.append(kv_seq)

    seq_len = kv_8_list[0].shape[0]

    ACTIVE_KV_8 = kv_8_list[0].unsqueeze(0)
    ACTIVE_KV_16 = kv_16_list[0].unsqueeze(0)
    ACTIVE_KV_24 = kv_24_list[0].unsqueeze(0)
    ACTIVE_MASK = torch.zeros((1, seq_len), dtype=torch.bool, device=device)

    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    sponge_id = tokenizer("...", add_special_tokens=False).input_ids[-1]

    sponge_matches = (inputs.input_ids[0] == sponge_id).nonzero(as_tuple=True)[0]
    ACTIVE_SPONGE_IDX = sponge_matches[-1].item() if len(sponge_matches) > 0 else inputs.input_ids.shape[1] - 1

    with torch.inference_mode(), torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        gens = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(gens[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    return generated_text


Loading best weights from best_multilayer_injector.pt...
Loading Datasets & Vectors...


# Cell 3: Execution

In [5]:
# --- EXECUTION & PRINTING ---
def run_inspection(sample_df, vecs_dict, split_name):
    print(f"\n{'='*80}")
    print(f"🕵️ VISUAL INSPECTION: {split_name} SET")
    print(f"{'='*80}\n")

    for i, row in sample_df.iterrows():
        print(f"--- Sample {i+1} | Task: {row['task_name']} ---")
        print(f"📄 ORIGINAL CONTEXT:\n{row['context']}\n")

        if row['task_name'] == "QA":
            q_part = row['inj_prompt'].split("Question: ")[-1].split("\n")[0]
            print(f"❓ QUESTION:\n{q_part}\n")

        print(f"🎯 GROUND TRUTH:\n{row['target_response']}\n")

        gen_resp = generate_steered_response(row, vecs_dict)
        print(f"🤖 GENERATED RESPONSE:\n{gen_resp}")
        print("-" * 80 + "\n")

run_inspection(train_sample, train_vecs, "TRAIN")
run_inspection(test_sample, test_vecs, "VALIDATION")

# h8.remove()
# h16.remove()
# h24.remove()

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



🕵️ VISUAL INSPECTION: TRAIN SET

--- Sample 1 | Task: Repeat ---
📄 ORIGINAL CONTEXT:
Picture the once-dilapidated neighborhood of Easthaven. Abandoned lots, neglected public spaces, and a general sense of decay described this corner of the city not too long ago. That's until the city council, led by Mayor Annalise Beckwith, strategized a public works program to breathe life into the failing infrastructure. Urban planners and community leaders like Jamal Richardson were tapped to create a comprehensive blueprint that addressed long-term sustainability and the immediate needs of the residents. The program focused on revitalizing parks, upgrading transportation systems, and constructing public amenities. The implementation date was set for October 1, 2023, and the community watched with a hopeful gaze as plans turned into tangible action.

🎯 GROUND TRUTH:
Picture the once-dilapidated neighborhood of Easthaven. Abandoned lots, neglected public spaces, and a general sense of decay describe